<a href="https://colab.research.google.com/github/Bigoal/flyrank-ml-internship/blob/main/work/Bigo_work/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigoal/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Hi For now I am going to make a research for 2 lanes and will decided in the future which one I will stick with

**Lane 2:**
Question: which pages should be queued for review based on signs like missed opportunity, deteriorates?

*reviewing all pages is an overhead task which is impossible to be done so rather than that we should flag pages for review based on their potential against their performance, then sort them based on opportunity/decline score.*


---


**Lane 4:**
Question: where and how much is the CTR gap between pages with the same position tier?

*high position tier and impression rate and a low ctr is a real big miss and means there is a large missed potential of these pages so they must be flagged with a score based on the gap with average CTR and how good is their position.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. The question: decision, action, cost of a wrong call
**Lane 2:**  
D: Which of the hundreds of pages a content reviewer deserves consuming the reviewer's fuel

A: Review these N pages as they are flagged as missed potential according to their score in  potential against their performance

C: Time waste on pages that got no potential or acceptable performance rather than pages which with a small review can do much better. Also missing other pages with high potential is a big waste.

---



**Lane 4:**  
D: Whether a specific page's underperforming CTR is worth acting on, given how much traffic is actually at stake at that position.

A: Review reasons for these pages based on their position tier and the gap between expected and real CTR  

C: Time waste and resources wasted on pages with no potential rather than pages with good content and high potential that only needs a good written title or repick the snippet part

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


30000 pages |  declining rate: 0.542


## Lane 2:

In [13]:
stale_visible_page = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
declining_with_demand = df[(df["trend_direction"] == "down" )& (df["impressions_90d"] >= 100)]
thin_visible_page= df[(df["word_count"] > 0)&  (df["word_count"] < 1200) &  (df["impressions_90d"] >= 250)]
page_one_decay_risk= df[(df["avg_position"] > 0) & (df["avg_position"] <= 10) &  (df["content_age_days"] >= 180)]
declining_with_demand_500 = df[(df["trend_direction"] == "down" )& (df["impressions_90d"] >= 500)]

visible_pool = df[df["impressions_90d"] >= 500]
print("Visible pool (impressions_90d >= 500):", len(visible_pool))

print("Count of stale but still visible pages:", len(stale_visible_page))
print("Count of pages declining despite real demand:", len(declining_with_demand))
print("Count of pages declining despite demand >500:", len(declining_with_demand_500),
      f"({len(declining_with_demand_500) / len(visible_pool):.1%} of visible pool)")
print("Count of pages with word count<1200 and visible:", len(thin_visible_page))
print("Count of old pages but still in top position:", len(page_one_decay_risk))


Visible pool (impressions_90d >= 500): 16726
Count of stale but still visible pages: 17
Count of pages declining despite real demand: 13152
Count of pages declining despite demand >500: 9961 (59.6% of visible pool)
Count of pages with word count<1200 and visible: 82
Count of old pages but still in top position: 7076


## Lane 4:

In [14]:
df_clean = df[df["avg_position"] > 0].copy()

eligible = df_clean[df_clean["impressions_90d"] >= 500].copy()

tier_median_eligible = eligible.groupby("position_tier")["ctr"].transform("median")
eligible["ctr_gap"] = eligible["ctr"] - tier_median_eligible

below_median = (eligible["ctr_gap"] < 0).sum()
meaningful_gap = eligible[eligible["ctr"] < 0.75 * tier_median_eligible]

print("Eligible pool:", len(eligible))
print("Below eligible-tier median:", below_median,
      f"({below_median / len(eligible):.1%} of eligible pool)")
print("Meaningfully below eligible-tier median (25%+ gap):", len(meaningful_gap),
      f"({len(meaningful_gap) / len(eligible):.1%} of eligible pool)")

Eligible pool: 16726
Below eligible-tier median: 7950 (47.5% of eligible pool)
Meaningfully below eligible-tier median (25%+ gap): 6339 (37.9% of eligible pool)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Lane 2:**

Can say:

* We observed that a meaningful share of visible pages show signs of staleness, decline, or thin content.
* These signals suggest certain pages are reasonable candidates for review, given limited reviewer time.
* Our score is decision-support — it ranks candidates, it doesn't replace a reviewer's judgment.

Can't say:

* This page is declining because of X.
* Refreshing this page will cause traffic to recover.
* Our model predicts what will happen to this page in the future.

**Lane 4:**
Can say:

* This page's CTR falls short of comparable pages at the same position, based on 90 days of observed data.
* The gap suggests the page may be under-capturing clicks relative to its peers.
* This is directional evidence for a reviewer to investigate, not a diagnosis.

Can't say:

* This page has a bad title or snippet (you don't know the cause — only that there's a gap).
* This gap is caused by X.
* Our analysis reflects how Google's ranking algorithm works, or predicts anything about it.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.